In [ ]:
import os
from tqdm import tqdm

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
import torch
import torch.optim as optim
from torch.utils.data import DataLoader

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score

In [ ]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(DEVICE)

# Датасет

In [ ]:
DATASET_NAME = "stanfordnlp/imdb"
MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 256
BATCH_SIZE = 16

dataset = load_dataset(DATASET_NAME)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=MAX_LENGTH)

tokenized_datasets = dataset.map(tokenize_function, batched=True)
tokenized_datasets = tokenized_datasets.remove_columns(["text"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")

train_dataset = tokenized_datasets["train"]
val_dataset = tokenized_datasets["test"]
unlabeled_dataset = tokenized_datasets["unsupervised"]

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(unlabeled_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}")
print(f"Unlabeled samples: {len(unlabeled_dataset)}")

# Модель

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(DEVICE)

optimizer = optim.AdamW(model.parameters(), lr=2e-5, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

In [ ]:
def train_epoch(loader):
    model.train()
    total_loss = 0
    all_preds =[]
    all_labels =[]

    pbar = tqdm(loader, desc='Training')
    for batch in pbar:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}

        optimizer.zero_grad()
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * batch['input_ids'].size(0)
        preds = torch.argmax(outputs.logits, dim=-1)
        all_preds.append(preds.cpu())
        all_labels.append(batch['labels'].cpu())

        pbar.set_postfix({'Loss': f'{total_loss / len(loader.dataset):.4f}'})

    all_preds = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()
    acc = accuracy_score(all_labels, all_preds)

    return total_loss / len(loader.dataset), acc

def validate(loader):
    model.eval()
    total_loss = 0
    all_preds =[]
    all_labels =[]

    with torch.no_grad():
        pbar = tqdm(loader, desc='Validation')
        for batch in pbar:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            outputs = model(**batch)
            loss = outputs.loss

            total_loss += loss.item() * batch['input_ids'].size(0)
            preds = torch.argmax(outputs.logits, dim=-1)
            all_preds.append(preds.cpu())
            all_labels.append(batch['labels'].cpu())

            pbar.set_postfix({'Loss': f'{total_loss / len(loader.dataset):.4f}'})

    all_preds = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()
    acc = accuracy_score(all_labels, all_preds)

    return total_loss / len(loader.dataset), acc

In [ ]:
EPOCHS = 3
best_val_acc = 0.0

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print("-" * 50)

    train_loss, train_acc = train_epoch(train_loader)
    val_loss, val_acc = validate(val_loader)

    scheduler.step()

    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_model.pth')
        print(f"Saved best model with val accuracy: {val_acc:.4f}")

print(f"\nBest validation accuracy: {best_val_acc:.4f}")

# Предсказания

In [ ]:
model.load_state_dict(torch.load('best_model.pth'))
model.eval()

predictions =[]

with torch.no_grad():
    pbar = tqdm(test_loader, desc='Predicting unsupervised data')
    for batch in pbar:
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=-1)

        predictions.extend(preds.cpu().numpy())

submission_df = pd.DataFrame({
    'id': range(len(predictions)),
    'text': dataset["unsupervised"]["text"],
    'label_id': predictions
})

LABEL_MAPPING = {0: 'negative', 1: 'positive'}
submission_df['prediction'] = submission_df['label_id'].map(LABEL_MAPPING)

submission_df.to_csv('submission.csv', index=False)
print("\nSubmission saved to submission.csv\n")

pd.set_option('display.max_colwidth', 256)
print(submission_df[['prediction', 'text']].head(15))